In [ ]:
import numpy as np
import rasterio
from rasterio.transform import Affine
from pyproj import Transformer
from scipy.interpolate import RegularGridInterpolator
import h5py
from rasterio.crs import CRS
from pathlib import Path
import re
import glob
from datetime import datetime
from typing import List, Tuple, Dict, Any
from mintpy.utils import utils as ut

In [ ]:
def write_mintpy_array_as_geotiff(
    input_array: np.ndarray,
    mintpy_timeseries_path: str,
    out_tif_path: str,
    nodata: float = np.nan,
    compress: str = "deflate",
    predictor: int = 3,          # 2 for int 3 for float
    bigtiff: str = "IF_SAFER",
    creation_overviews: bool = False,
    overview_levels = (2, 4, 8, 16),
    description: str = "Layer on MintPy geocoded grid",
    extra_tags: dict | None = None,
):
    """
    Save a 2D NumPy array to a GeoTIFF positioned on a MintPy geocoded grid (EPSG:4326).

    Parameters
    ----------
    input_array : (LENGTH, WIDTH) float/np.ndarray
        Data already aligned to MintPy geocoded grid.
    mintpy_timeseries_path : str
        Path to MintPy geocoded file (e.g., timeseries.h5) containing grid metadata.
    out_tif_path : str
        Output GeoTIFF path.
    nodata : float, default np.nan
        Nodata value to set in the GeoTIFF. np.nan is fine for float32.
    compress : str, default "deflate"
        GDAL compression (e.g., "deflate", "lzw", "zstd").
    predictor : int, default 3
        GDAL predictor (2 for horizontal, 3 for float-delta).
    bigtiff : str, default "IF_SAFER"
        BigTIFF policy ("YES","NO","IF_NEEDED","IF_SAFER").
    creation_overviews : bool, default False
        If True, build internal overviews at `overview_levels`.
    overview_levels : tuple, default (2,4,8,16)
        Overview decimation factors.
    description : str
        Written as band description and as a tag.
    extra_tags : dict | None
        Extra key/value metadata to store as dataset tags.
    """
    if input_array.ndim != 2:
        raise ValueError("input_array must be 2D (LENGTH x WIDTH).")

    length, width = input_array.shape

    # ---- read MintPy grid definition ----
    with h5py.File(mintpy_timeseries_path, "r") as f:
        attrs = dict(f.attrs.items())

        # Preferred: attributes provide regular grid definition
        if all(k in attrs for k in ("X_FIRST","X_STEP","Y_FIRST","Y_STEP","LENGTH","WIDTH")):
            try:
                x_first = float(attrs["X_FIRST"]); x_step = float(attrs["X_STEP"])
                y_first = float(attrs["Y_FIRST"]); y_step = float(attrs["Y_STEP"])
                L = int(attrs["LENGTH"]); W = int(attrs["WIDTH"])
            except Exception as e:
                raise ValueError(f"Unexpected MintPy attributes: {e}")

            if (L, W) != (length, width):
                raise ValueError(f"input_array shape {input_array.shape} != MintPy grid ({L},{W}).")

            transform = Affine(x_step, 0.0, x_first,
                               0.0,   y_step, y_first)
        else:
            # Fallback: try to reconstruct from latitude/longitude datasets
            if "latitude" not in f or "longitude" not in f:
                raise ValueError(
                    "MintPy file lacks X_* / Y_* attrs and latitude/longitude datasets. "
                    "Cannot derive GeoTIFF transform."
                )
            lat = f["latitude"][()]
            lon = f["longitude"][()]

            if lat.shape != input_array.shape or lon.shape != input_array.shape:
                raise ValueError("lat/lon shape must match input_array shape for fallback mode.")

            # Check near-regular grid (no rotation/skew). Use first row/col diffs.
            # Tolerate tiny floating noise.
            tol = 1e-9
            dx_row = np.nanmedian(np.diff(lon[0, :]))
            dy_col = np.nanmedian(np.diff(lat[:, 0]))
            # Validate consistency along rows/cols
            if not (np.allclose(np.diff(lon, axis=1), dx_row, atol=tol, equal_nan=True).all() and
                    np.allclose(np.diff(lat, axis=0), dy_col, atol=tol, equal_nan=True).all()):
                raise ValueError(
                    "Latitude/longitude grid is not a regular rectilinear grid; "
                    "cannot write a simple affine GeoTIFF transform."
                )

            x_first = float(lon[0, 0])
            y_first = float(lat[0, 0])
            x_step = float(dx_row)
            y_step = float(dy_col)
            transform = Affine(x_step, 0.0, x_first,
                               0.0,   y_step, y_first)

    # ---- write GeoTIFF ----
    dtype = np.float32 if np.issubdtype(input_array.dtype, np.floating) else input_array.dtype
    profile = {
        "driver": "GTiff",
        "height": length,
        "width": width,
        "count": 1,
        "dtype": dtype,
        "crs": CRS.from_epsg(4326),  # geocoded lat/lon
        "transform": transform,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "compress": compress,
        "predictor": predictor,
        "bigtiff": bigtiff,
        "nodata": nodata,
    }

    with rasterio.open(out_tif_path, "w", **profile) as dst:
        dst.write(input_array.astype(dtype, copy=False), 1)
        dst.set_band_description(1, description)

        # Tags: carry useful MintPy attrs for provenance
        tags = {
            "LENGTH": str(length),
            "WIDTH": str(width),
            "X_FIRST": str(x_first),
            "X_STEP": str(x_step),
            "Y_FIRST": str(y_first),
            "Y_STEP": str(y_step),
            "Description": description,
            "Source_MintPy_File": mintpy_timeseries_path,
        }
        if extra_tags:
            tags.update({str(k): str(v) for k, v in extra_tags.items()})
        # Also copy any other harmless scalar attrs
        for k in ("UTM_ZONE","REF_YEAR","REF_DATE","CENTER_LAT","CENTER_LON"):
            if k in attrs:
                tags[f"MintPy_{k}"] = str(attrs[k])
        dst.update_tags(**tags)

        if creation_overviews:
            dst.build_overviews(overview_levels, rasterio.enums.Resampling.average)
            dst.update_tags(ns="rio_overview", resampling="average")

    return out_tif_path


In [ ]:
def geotiff_to_mintpy_grid(geotiff_path: Path, mintpy_timeseries_path: Path, write_out=False) -> np.ndarray:
    """
    Interpolate a GeoTIFF (any projection) onto a MintPy geocoded lat/lon grid
    using SciPy RegularGridInterpolator.

    Parameters
    ----------
    geotiff_path : str
        Path to input GeoTIFF (single-band preferred).
    mintpy_timeseries_path : str
        Path to MintPy time series file (e.g., timeseries.h5). Must contain
        geocoded latitude/longitude grid either as datasets or via attributes
        (Y_FIRST, Y_STEP, X_FIRST, X_STEP, LENGTH, WIDTH).

    Returns
    -------
    np.ndarray
        2D array (LENGTH, WIDTH) with the GeoTIFF values interpolated onto
        the MintPy lat/lon grid. np.nan where outside coverage or nodata.
    """

    # ---- load MintPy target lat/lon grid ----
    with h5py.File(mintpy_timeseries_path, "r") as f:
        if "latitude" in f and "longitude" in f:
            lat = f["latitude"][()]
            lon = f["longitude"][()]
            # If stored as 1D vectors, expand to 2D
            if lat.ndim == 1 and lon.ndim == 1:
                lat = np.repeat(lat[:, None], lon.size, axis=1)
                lon = np.repeat(lon[None, :], lat.shape[0], axis=0)
        else:
            attrs = f.attrs
            def _geta(k, cast=float):
                v = attrs[k]
                if isinstance(v, (bytes, bytearray)):
                    v = v.decode()
                return cast(v)
            length = _geta("LENGTH", int)
            width  = _geta("WIDTH",  int)
            y_first = _geta("Y_FIRST", float)  # latitude row 0
            y_step  = _geta("Y_STEP",  float)  # often negative
            x_first = _geta("X_FIRST", float)  # longitude col 0
            x_step  = _geta("X_STEP",  float)
            lats = y_first + np.arange(length, dtype="float64") * y_step
            lons = x_first + np.arange(width,  dtype="float64") * x_step
            lat, lon = np.meshgrid(lats, lons, indexing="ij")

    # ---- open GeoTIFF ----
    with rasterio.open(geotiff_path) as ds:
        if ds.crs is None:
            raise ValueError("GeoTIFF has no CRS defined.")
        transform: Affine = ds.transform
        data = ds.read(1, masked=True).astype("float64")  # first band
        h, w = ds.height, ds.width
        # convert nodata mask to NaN (interpolator will propagate)
        data = np.where(np.ma.getmaskarray(data), np.nan, data)

        # build pixel index axes for RegularGridInterpolator
        row_axis = np.arange(h, dtype="float64")
        col_axis = np.arange(w, dtype="float64")

        # transform MintPy (lon,lat) -> GeoTIFF CRS (x,y)
        to_src = Transformer.from_crs("EPSG:4326", ds.crs, always_xy=True)
        x_flat, y_flat = to_src.transform(lon.ravel(), lat.ravel())

        # convert x,y -> fractional (row,col) via inverse affine (handles rotation/skew)
        A = np.array([[transform.a, transform.b, transform.c],
                      [transform.d, transform.e, transform.f],
                      [0.0,         0.0,         1.0        ]], dtype="float64")
        Ainv = np.linalg.inv(A)
        XY1 = np.vstack((x_flat, y_flat, np.ones_like(x_flat)))
        CR1 = Ainv @ XY1
        cols_f = CR1[0, :]
        rows_f = CR1[1, :]

        # set up interpolator (row, col) ordering
        rgi = RegularGridInterpolator(
            (row_axis, col_axis),
            data,
            method="linear",
            bounds_error=False,
            fill_value=np.nan,
        )

        out = rgi(np.column_stack([rows_f, cols_f])).reshape(lat.shape)
        with h5py.File(mintpy_timeseries_path, "r") as f:
            L, W = int(f.attrs["LENGTH"]), int(f.attrs["WIDTH"])
        assert out.shape == (L, W), f"Got {out.shape}, expected {(L, W)}"
        if write_out == True:
            outname = 'resampled_' + geotiff_path.stem + geotiff_path.suffix
            write_mintpy_array_as_geotiff(out, mintpy_timeseries_path, outname,
                                          creation_overviews=True,
                                          description="LIDAR resampled to MintPy geocoded grid")
            return print(f'Wrote resampled data {outname}')
        else:
            return out

In [ ]:
ts_f = Path('/Users/havazli/Desktop/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/CA_P67_F750_A_Dec22_June23/mintpy/geo/geo_timeseries_ERA5.h5')
aso_folder = Path('/Users/havazli/Desktop/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/ASO')
year = '2023'
aso_files = list(aso_folder.joinpath(year).rglob('*swe*.tif'))

In [ ]:
for lidar_file in aso_files:
    geotiff_to_mintpy_grid(lidar_file, ts_f, write_out=True)

In [ ]:
# Matches YYYYMonDD or YYYYMonDD-DD (e.g., 2023Apr09 or 2023May11-12)
_PAT = re.compile(r"(\d{4}[A-Za-z]{3}\d{2})(?:-(\d{2}))?")

def _extract_start_date_str(p: str) -> str:
    """Extract start date as YYYYMMDD from a path string."""
    m = _PAT.search(p)
    if not m:
        raise ValueError(f"No date token found in: {p}")
    return datetime.strptime(m.group(1).title(), "%Y%b%d").strftime("%Y%m%d")

def read_geotiff_stack_sorted_by_date(
    pattern: str,
    dtype: str = "float32",
    strict_same_grid: bool = True,
) -> Tuple[List[str], np.ndarray, Dict[str, Any]]:
    """
    Glob GeoTIFFs, extract start dates from names, sort by date, and read into a 3D array.

    Parameters
    ----------
    pattern : str
        Glob pattern (e.g., "/path/**/ASO_*_snowdepth_3m-*.tif").
        Wildcards and ** (recursive) are supported.
    dtype : str, default "float32"
        Output dtype for the data array.
    strict_same_grid : bool, default True
        If True, require all rasters to have identical shape, transform, and CRS.
        If False, only shape must match (you can resample later if needed).

    Returns
    -------
    dates : List[str]
        Start dates as "YYYYMMDD", sorted ascending.
    data : np.ndarray
        Array of shape (T, H, W) with NaNs where nodata; dtype per `dtype`.
    meta : Dict[str, Any]
        Metadata from the first file: {"profile", "transform", "crs", "paths"}.
    """
    # 1) Collect files
    paths = sorted(glob.glob(pattern, recursive=True))
    if not paths:
        raise FileNotFoundError(f"No files matched pattern: {pattern}")

    # 2) Extract start dates and sort
    items = []
    for p in paths:
        d = _extract_start_date_str(p)
        items.append((d, p))
    items.sort(key=lambda x: x[0])  # by YYYYMMDD string

    dates = [d for d, _ in items]
    sorted_paths = [p for _, p in items]

    # 3) Read first file to define grid
    with rasterio.open(sorted_paths[0]) as ds0:
        h, w = ds0.height, ds0.width
        transform0 = ds0.transform
        crs0 = ds0.crs
        profile0 = ds0.profile.copy()

    # 4) Preallocate stack
    T = len(sorted_paths)
    data = np.full((T, h, w), np.nan, dtype=np.dtype(dtype))

    # 5) Read all files and verify grid consistency
    for t, p in enumerate(sorted_paths):
        with rasterio.open(p) as ds:
            if ds.height != h or ds.width != w:
                raise ValueError(f"Shape mismatch: {p} has {(ds.height, ds.width)} != {(h, w)}")
            if strict_same_grid:
                if ds.crs != crs0:
                    raise ValueError(f"CRS mismatch: {p} has {ds.crs} != {crs0}")
                if not np.allclose(ds.transform, transform0):
                    raise ValueError(f"Transform mismatch for {p}")

            band = ds.read(1, masked=True)  # honors nodata
            arr = np.where(np.ma.getmaskarray(band), np.nan, band.astype(dtype, copy=False))
            data[t] = arr

    return dates, data


In [ ]:
date_list, stack = read_geotiff_stack_sorted_by_date('resampled_ASO*')

num_date = len(date_list)
meta = ut.readfile.read_attribute(ts_f)
_ = meta.pop('REF_DATE')
_ = meta.pop('REF_LAT')
_ = meta.pop('REF_LON')
_ = meta.pop('REF_X')
_ = meta.pop('REF_Y')

In [ ]:
dates = np.array(date_list, np.bytes_)
ds_name_dict = {
    "date"       : [np.dtype("S8"), (num_date,), dates],
    "bperp"      : [np.float32,     (num_date,), None],
    "timeseries" : [np.float32,     (num_date, stack.shape[1], stack.shape[2]), stack],
}
ut.writefile.layout_hdf5('timeseries_lidar_swe.h5', ds_name_dict, metadata=meta)